In [6]:
import spatialdata as sd
import decoupler as dc
from spatialdata_io import xenium

import numpy as np
import pandas as pd
import scipy.sparse as sp


import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad

import scanpy as sc

import squidpy as sq

sc.set_figure_params(figsize=(8,8))
mpl.rcParams['figure.dpi'] = 300
mpl.rcParams['font.size'] = 10              # Universal font size
mpl.rcParams['axes.titlesize'] = 10          # Title size
mpl.rcParams['axes.labelsize'] = 10          # X and Y label size
mpl.rcParams['xtick.labelsize'] = 10         # X tick label size
mpl.rcParams['ytick.labelsize'] = 10        # Y tick label size
mpl.rcParams['legend.fontsize'] = 10         # Legend font size
mpl.rcParams['font.family'] = 'DejaVu Sans'  # or 'Arial'
mpl.rcParams['svg.fonttype'] = 'none'

import os

import sys
plt.style.use('default')

In [7]:
Sample_895=sc.read_h5ad("/exports/archive/hg-exon-skip/Qirong/Xenium/scanpy_object/Sample_895.h5ad")
Sample_894=sc.read_h5ad("/exports/archive/hg-exon-skip/Qirong/Xenium/scanpy_object/Sample_894.h5ad")
Sample_898=sc.read_h5ad("/exports/archive/hg-exon-skip/Qirong/Xenium/scanpy_object/Sample_898.h5ad")


Sample_896=sc.read_h5ad("/exports/archive/hg-exon-skip/Qirong/Xenium/scanpy_object/Sample_896.h5ad")
Sample_897=sc.read_h5ad("/exports/archive/hg-exon-skip/Qirong/Xenium/scanpy_object/Sample_897.h5ad")
Sample_18=sc.read_h5ad("/exports/archive/hg-exon-skip/Qirong/Xenium/scanpy_object/Sample_18.h5ad")

Sample_886=sc.read_h5ad("/exports/archive/hg-exon-skip/Qirong/Xenium/scanpy_object/Sample_886.h5ad")
Sample_890=sc.read_h5ad("/exports/archive/hg-exon-skip/Qirong/Xenium/scanpy_object/Sample_890.h5ad")
Sample_891=sc.read_h5ad("/exports/archive/hg-exon-skip/Qirong/Xenium/scanpy_object/Sample_891.h5ad")


Sample_4=sc.read_h5ad("/exports/archive/hg-exon-skip/Qirong/Xenium/scanpy_object/Sample_4.h5ad")
Sample_5=sc.read_h5ad("/exports/archive/hg-exon-skip/Qirong/Xenium/scanpy_object/Sample_5.h5ad")
Sample_3=sc.read_h5ad("/exports/archive/hg-exon-skip/Qirong/Xenium/scanpy_object/Sample_3.h5ad")


In [8]:
adata = Sample_896.concatenate(Sample_897,Sample_18,Sample_895,Sample_894,Sample_898)

/tmp/ipykernel_5335/1233193766.py:1: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata = Sample_896.concatenate(Sample_897,Sample_18,Sample_895,Sample_894,Sample_898)


In [4]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [8]:
# Create a key combining Sample + cell_type + region
adata.obs['Sample_CellType_Region'] = (
    adata.obs['Sample'].astype(str) + "_" + adata.obs['Class'].astype(str) + "_" + adata.obs['Region'].astype(str)
)

# Group cells by this combined key and get indices
grouped = adata.obs.groupby('Sample_CellType_Region').indices

# Aggregate mean expression per group
rep_means_list = []
for key, indices in grouped.items():
    # mean expression for this group
    mean_expr = np.asarray(adata[indices].X.mean(axis=0)).ravel()  # convert sparse to dense if needed
    rep_means_list.append(
        pd.DataFrame([mean_expr], columns=adata.var_names)
        .assign(Sample_CellType_Region=key)
    )

rep_means = pd.concat(rep_means_list, ignore_index=True)

# Split the combined key back to columns
rep_means[['Sample', 'cell_type', 'region']] = rep_means['Sample_CellType_Region'].str.split("_", expand=True)
rep_means.drop(columns='Sample_CellType_Region', inplace=True)
sample_to_group = adata.obs[['Sample', 'Group']].drop_duplicates().set_index('Sample')['Group']
rep_means['Group'] = rep_means['Sample'].map(sample_to_group)

# Now melt to long format
long_df = rep_means.melt(id_vars=['Sample', 'cell_type', 'region'], var_name='Gene', value_name='expression')

# Map Group from adata.obs
long_df['Group'] = long_df['Sample'].map(sample_to_group)

In [5]:
gene = "Dystrophin_Exon2_29"

mean_expr = rep_means.groupby(['cell_type', 'region', 'Group'])[gene].mean().reset_index()

NameError: name 'rep_means' is not defined

In [ ]:
pivot_df = mean_expr.pivot_table(
    index=['cell_type', 'region'],
    columns='Group',
    values=gene
).fillna(0)

In [ ]:
pivot_df['delta'] = pivot_df['mdx52'] - pivot_df['WT']

In [ ]:
heatmap_data = pivot_df['delta'].unstack(level='cell_type')

In [ ]:
celltype_order = adata.obs['Class'].cat.categories.tolist()
region_order = adata.obs['Region'].cat.categories.tolist()

heatmap_data = heatmap_data.reindex(columns=celltype_order, index=region_order)

In [ ]:
heatmap_data 

In [ ]:
counts = pd.crosstab(adata.obs["Region"], adata.obs["Class"])

In [ ]:
counts 

In [ ]:
total_cells_per_region = counts.sum(axis=0)
threshold_per_region = total_cells_per_region * 0.005

In [ ]:
threshold_per_region 

In [ ]:
mask_1

In [ ]:
mask_1 = counts.gt(threshold_per_region, axis=1)

filtered_heatmap_data = heatmap_data.where(mask_1)

In [ ]:
filtered_heatmap_data

In [ ]:
mpl.rcParams['font.family'] = 'DejaVu Sans'  # or 'Arial'
mpl.rcParams['svg.fonttype'] = 'none'

# Update rcParams for consistent formatting
plt.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.size': 6,
    'axes.titlesize': 6,
    'axes.labelsize': 6,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'legend.fontsize': 6,
})


heatmap_data=heatmap_data.fillna(0)


# Plot heatmap
plt.figure(figsize=(3.2, 3.1))

sns.heatmap(
    filtered_heatmap_data,
    cmap='bwr',    # blue = negative, white = zero, red = positive
    center=0,
    annot=False
)

plt.ylabel("Region")
plt.xlabel("Class")

plt.tight_layout()
plt.savefig("/exports/humgen/qmao/Dystrophin_Exon2_29_delta_expression.svg",format="svg",dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Specify the gene
gene = "Dp427c"

# Check if gene is in adata
if gene not in adata.var_names:
    raise ValueError(f"{gene} not found in adata.var_names")

# Get expression values for gene1
gene_expr = adata[:, gene].X

# Convert to 1D array
if hasattr(gene_expr, "toarray"):
    gene_expr = gene_expr.toarray().flatten()
else:
    gene_expr = np.array(gene_expr).flatten()

# Create a boolean mask: True if gene1 > 0
positive_mask = gene_expr > 0

# Add to obs
adata.obs["gene1_positive"] = positive_mask

# Crosstab: counts of gene1-positive cells per (Region, Class)
positive_counts = pd.crosstab(
    adata.obs["Region"],
    adata.obs["Class"],
    values=adata.obs["gene1_positive"],
    aggfunc="sum",
    dropna=False
).fillna(0).astype(int)

positive_counts